<!-- STEP A0 -->
## Setup: data, split, models and shared helpers

This section does what the lab's Section 4 asks: reuse the cleaned CICIDS data from Lab 1, rebuild the
same train / validation / test split, and train the three models (decision tree, logistic regression,
random forest of 300 trees) with `random_state=42`.

It also builds every shared name the later steps use (see the contract in `TASKLIST.md`, section 2.3):
the feature lists, the training standard deviations and medians, the correlation table, `report()`,
and the two fixed samples of test flows (`ATTACK_IDX`, `EXPLAIN_IDX`).

**Data note.** Lab 1 used CICIDS**2017** (all eight day-files, 20% stratified sample), not CICIDS2018
as the lab text says. The column names are therefore the long 2017 ones (`Fwd Packet Length Mean`
rather than `Fwd Pkt Len Mean`), about 15% of flows are attacks, and `Destination Port` and
`Protocol` were removed in Lab 1 as ID columns.

In [1]:
# STEP A0
# Install shap and lime only if they are missing (Colab). Locally they come from requirements.txt.
import importlib.util
import subprocess
import sys

_missing = [p for p in ("shap", "lime") if importlib.util.find_spec(p) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

In [2]:
# STEP A0
import os
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix, f1_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 42

# scikit-learn 1.6.1 passes an option that newer scipy no longer knows; the warning is harmless.
warnings.filterwarnings("ignore", message="Unknown solver options")

# Part notebooks live in parts/; the hand-in notebook lives in the repository root. Work from the
# root in both cases, so that data/, models/ and results/ mean the same thing everywhere.
if Path.cwd().name == "parts":
    os.chdir("..")

DATA_FILE = Path("data/processed/clean.csv")
if not DATA_FILE.is_file() and Path("clean.csv").is_file():
    DATA_FILE = Path("clean.csv")              # Colab: clean.csv uploaded next to the notebook
MODEL_DIR = Path("models")
for folder in (MODEL_DIR, Path("results/figures"), Path("results/tables")):
    folder.mkdir(parents=True, exist_ok=True)

print(f"Python {sys.version.split()[0]}, scikit-learn {sklearn.__version__}, "
      f"numpy {np.__version__}, pandas {pd.__version__}")
print(f"Working directory: {Path.cwd()}")

Python 3.13.7, scikit-learn 1.6.1, numpy 2.5.3, pandas 3.0.6
Working directory: /home/steven/Desktop/AI-For-Cybersecurity-Lab-4.2-Robustness-attacks


<!-- STEP A0 -->
### Load the data and rebuild the Lab 1 split

The split is exactly Lab 1's (`lab1_pipeline/prepare.py`): 60 / 20 / 20, stratified on the **attack
type** so that every attack family keeps its share in all three parts, seed 42. The assertions stop
the notebook if the split is not the one Lab 1 used.

In [3]:
# STEP A0
if not DATA_FILE.is_file():
    raise FileNotFoundError(
        f"Cannot find {DATA_FILE}. Copy clean.csv into data/processed/ (see data/README.md).")

df = pd.read_csv(DATA_FILE)
attack_type = df["Label"]                               # "BENIGN", "DoS Hulk", "DDoS", ...
y = (attack_type != "BENIGN").astype(int)               # 1 = attack, 0 = benign
X = df.drop(columns=["Label"]).select_dtypes(include=[np.number])

# Same two cuts as Lab 1: first the test set (20%), then validation = 25% of the remaining 80%.
X_tmp, X_test, y_tmp, y_test, type_tmp, type_test = train_test_split(
    X, y, attack_type, test_size=0.20, stratify=attack_type, random_state=SEED)
X_train, X_val, y_train, y_val, type_train, type_val = train_test_split(
    X_tmp, y_tmp, type_tmp, test_size=0.25, stratify=type_tmp, random_state=SEED)
del X_tmp, y_tmp, type_tmp

assert X_train.shape == (267_984, 68), X_train.shape
assert X_val.shape == (89_328, 68), X_val.shape
assert X_test.shape == (89_329, 68), X_test.shape
assert int(y_test.sum()) == 13_461, int(y_test.sum())

for name, part, labels in (("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)):
    print(f"{name:<5} {len(part):>7,} rows   attack rate {labels.mean():.4f}")

train 267,984 rows   attack rate 0.1507
val    89,328 rows   attack rate 0.1507
test   89,329 rows   attack rate 0.1507


<!-- STEP A0 -->
### Shared feature information (measured on the training set only)

- `CONT_COLS` / `FLAG_COLS`: continuous columns, and the 0/1 flag columns (two or fewer distinct
  values). Noise in B1 is added to continuous columns only.
- `TRAIN_STD`, `TRAIN_MEDIAN`: the size of the noise (B1) and of the attacker's step (C1), and the
  value used for a missing feature (B2, F1). They come from the **training** set; measuring them on
  the test set would be leakage.
- `CORR`: absolute correlation between every pair of features, used to find the "twins" in E2 and
  F1.

In [4]:
# STEP A0
FEATURES = list(X_train.columns)
FLAG_COLS = [c for c in FEATURES if X_train[c].nunique() <= 2]
CONT_COLS = [c for c in FEATURES if c not in FLAG_COLS]

TRAIN_STD = X_train.std()
TRAIN_MEDIAN = X_train.median()
# Same numbers as X_train.corr().abs() (checked: differences below 1e-13), but about 7x faster.
CORR = pd.DataFrame(np.corrcoef(X_train.to_numpy(dtype=float), rowvar=False),
                    index=FEATURES, columns=FEATURES).abs()

print(f"{len(FEATURES)} features: {len(CONT_COLS)} continuous, {len(FLAG_COLS)} flag columns")
print("Flag columns:", ", ".join(FLAG_COLS))

68 features: 58 continuous, 10 flag columns
Flag columns: Fwd PSH Flags, Fwd URG Flags, FIN Flag Count, SYN Flag Count, RST Flag Count, PSH Flag Count, ACK Flag Count, URG Flag Count, CWE Flag Count, ECE Flag Count


<!-- STEP A0 -->
### `report()`: the scores the lab asks for

The lab says to always report macro-F1, recall, PR-AUC and FAR. `report()` returns those plus accuracy
and ROC-AUC, so the misleading scores can be shown next to the honest ones.

- **FAR** (false alarm rate) = FP / (FP + TN): the share of benign flows wrongly called attacks.
- **PR-AUC** = `average_precision_score`. Unlike ROC-AUC it is not inflated by the many benign flows.

It works with any model that has `predict` and `predict_proba` (the ensemble in E1 included).

In [5]:
# STEP A0
def false_alarm_rate(y_true, y_pred):
    """FP / (FP + TN): the share of benign flows that were called attacks."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0


def report(model, X, y):
    """Score a model on (X, y). X must be a DataFrame with the 68 raw feature columns."""
    y_pred = model.predict(X)
    y_score = model.predict_proba(X)[:, 1]
    return {
        "accuracy": accuracy_score(y, y_pred),
        "macro_f1": f1_score(y, y_pred, average="macro"),
        "recall": recall_score(y, y_pred, pos_label=1, zero_division=0),
        "roc_auc": roc_auc_score(y, y_score),
        "pr_auc": average_precision_score(y, y_score),
        "far": false_alarm_rate(y, y_pred),
    }

<!-- STEP A0 -->
### The three models

All three take the **raw** 68 columns. Logistic regression needs scaled features, so the scaler sits
inside a `Pipeline` and is fitted on the training set only. That way noise, medians and attacker steps
are always applied in real units, to every model in the same way.

| Model | Settings | Why |
|---|---|---|
| Decision tree | `random_state=42`, no depth limit | The single, brittle model the lab contrasts with the forest |
| Logistic regression | `StandardScaler` + `LogisticRegression(max_iter=1000)` | A linear model with a different decision boundary |
| Random forest | 300 trees, `max_depth=20`, `n_jobs=-1` | 300 trees as the lab says; depth 20 is what Lab 1 chose on validation, and it keeps SHAP fast |

Training the forest takes a few minutes, so each model is saved in `models/` and loaded on later
runs. **Delete `models/*.joblib` to retrain from scratch** (the final check, T7, does this).

In [6]:
# STEP A0
MODEL_SPECS = {
    "tree": lambda: DecisionTreeClassifier(random_state=SEED),
    "logreg": lambda: Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=SEED)),
    ]),
    "forest": lambda: RandomForestClassifier(n_estimators=300, max_depth=20,
                                             random_state=SEED, n_jobs=-1),
}


def load_or_train(name):
    """Load models/<name>.joblib if it fits the current features; otherwise train and save it."""
    path = MODEL_DIR / f"{name}.joblib"
    if path.is_file():
        model = joblib.load(path)
        if list(getattr(model, "feature_names_in_", [])) == FEATURES:
            print(f"  {name:<7} loaded from {path}")
            return model
        print(f"  {name:<7} cached file does not match the features; retraining")
    start = time.time()
    model = MODEL_SPECS[name]().fit(X_train, y_train)
    joblib.dump(model, path)
    print(f"  {name:<7} trained in {time.time() - start:.0f} s, saved to {path}")
    return model


MODELS = {name: load_or_train(name) for name in MODEL_SPECS}
tree, logreg, rf = MODELS["tree"], MODELS["logreg"], MODELS["forest"]

  tree    loaded from models/tree.joblib
  logreg  loaded from models/logreg.joblib


  forest  loaded from models/forest.joblib


<!-- STEP A0 -->
### Fixed samples of test flows, shared by several steps

- `ATTACK_IDX`: the **50 test flows the forest is most confident are attacks**, among real attacks it
  detects. C2 attacks these flows, E1 attacks them again with the ensemble, and F1 runs the deletion
  test on them, so all three talk about the same flows.
- `EXPLAIN_IDX`: 500 random test flows (seed 42). D1 explains all 500 with SHAP, D2 uses the first 200.

Both hold **positions**, so use them with `X_test.iloc[...]`.

In [7]:
# STEP A0
rf_proba_test = rf.predict_proba(X_test)[:, 1]
detected = np.flatnonzero((y_test.to_numpy() == 1) & (rf_proba_test >= 0.5))
# Highest probability first; "stable" keeps ties in test-set order, so the choice is reproducible.
ATTACK_IDX = detected[np.argsort(-rf_proba_test[detected], kind="stable")[:50]]

EXPLAIN_IDX = np.random.default_rng(SEED).choice(len(X_test), 500, replace=False)

print(f"ATTACK_IDX: 50 detected attacks, forest probability "
      f"{rf_proba_test[ATTACK_IDX].min():.3f}-{rf_proba_test[ATTACK_IDX].max():.3f}")
print("  attack types:", type_test.iloc[ATTACK_IDX].value_counts().to_dict())
print(f"EXPLAIN_IDX: {len(EXPLAIN_IDX)} random test flows, "
      f"{int(y_test.iloc[EXPLAIN_IDX].sum())} of them attacks")

ATTACK_IDX: 50 detected attacks, forest probability 1.000-1.000
  attack types: {'DDoS': 25, 'DoS Hulk': 22, 'DoS GoldenEye': 2, 'SSH-Patator': 1}
EXPLAIN_IDX: 500 random test flows, 68 of them attacks


In [8]:
# STEP A0
# Everything the later steps rely on (TASKLIST.md, section 2.3). Fail here, not halfway through Part F.
_shared = ["SEED", "X_train", "X_val", "X_test", "y_train", "y_val", "y_test", "FEATURES",
           "CONT_COLS", "FLAG_COLS", "TRAIN_STD", "TRAIN_MEDIAN", "CORR", "report", "MODELS",
           "tree", "logreg", "rf", "ATTACK_IDX", "EXPLAIN_IDX"]
missing = [name for name in _shared if name not in globals()]
assert not missing, f"setup did not create: {missing}"
print(f"Setup complete: {len(_shared)} shared names ready. The models are scored in step A2.")

Setup complete: 20 shared names ready. The models are scored in step A2.
